# Test Case for Perturb
Trying to implement perturb like Sara did

In [1]:
# Setup
import sys
import traceback
import torch
import json
import wandb
import time
import csv
import matplotlib
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import issparse

sys.path.append("../")
import scgpt as scg
from scgpt.model import TransformerModel
from scgpt.utils import set_seed, category_str2int, eval_scib_metrics, load_pretrained
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor
from torchtext._torchtext import (
    Vocab as VocabPybind,
)


#########################################
# Temporary
hyperparameter_defaults={}
hyperparameter_defaults['perturbation'] = dict(
    seed=0,
    #dataset_name="ms",
    do_train=True,
    #load_model="../save/scGPT_human",
    mask_ratio=0.0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=1e-4,
    batch_size=32,
    layer_size=128,
    nlayers=4,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=4,  # number of heads in nn.MultiheadAttention
    dropout=0.2,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
    GEPC = True
)

config = hyperparameter_defaults['perturbation']

config["dataset_name"] = "AML_All_Samples.h5ad"
config["model"] = "scGPT"
config["load_model"] = "../save/scGPT_kidney"
config["ft_model"] = "./save/dev_AML-Apr17-11-45/best_model.pt"
config["project"] = "scGPT"
config["batch_column"] = "ID"
config["celltype_column"] = "AnnoCellType"
config["condition_column"] = "disease"
config["celltype"]="LEC"
set_seed(config["seed"])

# settings for input and preprocessing
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
mask_ratio = config["mask_ratio"]
mask_value = -1
pad_value = -2
n_input_bins = config["n_bins"]

n_hvg = 1200  # number of highly variable genes
max_seq_len = n_hvg + 1
per_seq_batch_sample = True
DSBN = False  # Domain-spec batchnorm
explicit_zero_prob = True  # whether explicit bernoulli for zeros



#########################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## Paths ##
data_path = Path("../data/", config["dataset_name"])
save_dir = Path(f"./save/dev-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"saving to {save_dir}")

ft_model_path = Path(config["ft_model"])
pre_model_path = Path(config["load_model"])
## WandB ##

# run = wandb.init(
#     config=config,
#     project=config["project"],
#     reinit=True,
#     #mode="disabled",
#     settings=wandb.Settings(start_method="fork"),
# )
# config = wandb.config
# print(config)

## Logger ##
logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")
# log running date and current git commit
logger.info(f"Running on {time.strftime('%Y-%m-%d %H:%M:%S')}")



/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not inst

saving to save/dev-May26-12-13
scGPT - INFO - Running on 2026-05-26 12:13:38


# Plan
### Object Work
* Read in object
* Summarize object by condition
    * Number of cells
    * Gene frequency
* Preprocess object
* Tokenize object

### Model Work
* Load model 

###

In [2]:
# WIP - Object work

## Load data and standardize metadata
adata = sc.read_h5ad(data_path)

adata.obs["celltype"] = adata.obs[config["celltype_column"]].astype("category")
adata.obs["condition"] = adata.obs[config["condition_column"]].astype("category")
raw_data = adata.raw.X.copy()
if issparse(raw_data):
    # Only modify the non-zero data points for efficiency
    raw_data.data = np.round(raw_data.data).astype(int)
else:
    raw_data = np.round(raw_data).astype(int)
adata.layers["counts"] = raw_data
adata.X = adata.layers["counts"].copy()
#adata.var = adata.var.set_index("var.features")
data_is_raw = True

# make the batch category column
adata.obs["str_batch"] = adata.obs[config["batch_column"]].astype(str)
batch_id_labels = adata.obs["str_batch"].astype("category").cat.codes.values
adata.obs["batch_id"] = batch_id_labels

adata.var["gene_name"] = adata.var.index.tolist()

## Load model and filter data object to only include genes in vocab file

if config["load_model"] is not None:
    model_dir = Path(config["load_model"])
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])
    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    adata = adata[:, adata.var["id_in_vocab"] >= 0]

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    logger.info(
        f"Resume model from {model_file}, the model args will be overriden by the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]
else:
    embsize = config["layer_size"]
    nhead = config["nhead"]
    nlayers = config["nlayers"]
    d_hid = config["layer_size"]





## Preprocess the data
preprocessor = Preprocessor(
    use_key="X",  # the key in adata.layers to use as raw data
    filter_gene_by_counts=3,  # step 1
    filter_cell_by_counts=False,  # step 2
    normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
    result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
    log1p=data_is_raw,  # 4. whether to log1p the normalized data
    result_log1p_key="X_log1p",
    subset_hvg=n_hvg,  # 5. whether to subset the raw data to highly variable genes
    hvg_flavor="seurat_v3" if data_is_raw else "cell_ranger",
    binning=config["n_bins"],  # 6. whether to bin the raw data and to what number of bins
    result_binned_key="X_binned",  # the key in adata.layers to store the binned data
)
preprocessor(adata, batch_key="str_batch" if config["dataset_name"] != "heart_cell" else None)

if per_seq_batch_sample:
    # sort the adata by batch_id in advance
    adata_sorted = adata[adata.obs["batch_id"].argsort()].copy()



/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/anndata/compat/__init__.py:329: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


scGPT - INFO - match 30766/38851 genes in vocabulary of size 60697.
scGPT - INFO - Resume model from ../save/scGPT_kidney/best_model.pt, the model args will be overriden by the config ../save/scGPT_kidney/args.json.
scGPT - INFO - Filtering genes by counts ...


/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:281: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_counts"] = number


scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Log1p transforming ...
scGPT - INFO - Subsetting highly variable genes ...
scGPT - INFO - Binning data ...


In [3]:
from torchtext.vocab import Vocab
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value

input_layer_key = "X_binned"
all_counts = (
    adata.layers[input_layer_key].A
    if issparse(adata.layers[input_layer_key])
    else adata.layers[input_layer_key]
)
genes = adata.var["gene_name"].tolist()

celltypes_labels = adata.obs['celltype'].tolist()  # make sure count from 0
num_types = len(set(celltypes_labels))
celltypes_labels = np.array(celltypes_labels)

batch_ids = adata.obs["batch_id"].tolist()
num_batch_types = len(set(batch_ids))
batch_ids = np.array(batch_ids)

if config["load_model"] is None:
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(vocab(genes), dtype=int)

TOKENS = tokenize_and_pad_batch(
    all_counts,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=True,
)

In [4]:
# WIP - Model work
batch_ids = adata.obs["batch_id"].tolist()
num_batch_types = len(set(batch_ids))
batch_ids = np.array(batch_ids)
ntokens = len(vocab)
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    vocab=vocab,
    #dropout=model_configs["dropout"],
    pad_token=pad_token,
    pad_value=pad_value,
    #do_mvc=config["GEPC"],
    #do_dab=True,
    #use_batch_labels=False,
    #num_batch_labels=num_batch_types,
    #domain_spec_batchnorm=DSBN,
    n_input_bins=n_input_bins,
    # ecs_threshold=config["ecs_thres"],
    # explicit_zero_prob=explicit_zero_prob,
    # use_fast_transformer=config["fast_transformer"],
    # pre_norm=config["pre_norm"],
    cell_emb_style="cls"
)

model.to(device)


# ckpt = torch.load(config["ft_model"], map_location=device)

# state_dict = {
#     k.replace("self_attn.Wqkv", "self_attn.in_proj"): v
#     for k, v in ckpt.items()
# }
# model.load_state_dict(state_dict, strict=True)
model.eval()


TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(60697, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.5, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwis

In [5]:
#
import json
vocab = json.load(open(vocab_file)) 
# ===== SELECT CELLS =====
mask_lam = (adata.obs['celltype'] == config['celltype']) & (adata.obs["condition"] == "AML")
mask_ctrl = (adata.obs["celltype"] == config["celltype"]) & (adata.obs["condition"] == "Normal")

idx_lam = mask_lam.to_numpy().nonzero()[0]
idx_ctrl = mask_ctrl.to_numpy().nonzero()[0]

idx_lam_to_pos = {int(idx): i for i, idx in enumerate(idx_lam)}
idx_ctrl_to_pos = {int(idx): i for i, idx in enumerate(idx_ctrl)}

# ===== TOKEN MAPPINGS =====
token_to_gene = {int(v): k for k, v in vocab.items()}
gene_to_token = {v: k for k, v in token_to_gene.items()}




In [6]:
TOKENS = {
    i: {"genes": g, "values": v}
    for i, (g, v) in enumerate(zip(TOKENS["genes"], TOKENS["values"]))
}

In [7]:
# ===== EMBEDDING FUNCTION =====
@torch.no_grad()
def embed(indices, bs=64, min_bs=1):
    import torch, gc
    outs = []
    i = 0
    while i < len(indices):
        current_bs = min(bs, len(indices) - i)
        while True:
            try:
                batch_idx = indices[i:i+current_bs]
                toks = [TOKENS[j] for j in batch_idx]
                # Use all genes in the cell (Whole Genome)
                max_len = max(len(t["genes"]) for t in toks)
                G = torch.full((len(toks), max_len), pad_id, dtype=torch.long)
                V = torch.zeros((len(toks), max_len), dtype=torch.float16)
                for k, t in enumerate(toks):
                    g = t["genes"]
                    v = t["values"]
                    L = len(g)
                    if L > 0:
                        G[k, :L] = g
                        V[k, :L] = v if v.dtype == torch.float16 else v.to(torch.float16)
                M = (G == pad_id)
                g = G.to(device, non_blocking=True)
                v = V.to(device, non_blocking=True)
                m = M.to(device, non_blocking=True)
                with torch.no_grad(), torch.autocast(device_type="cuda"):
                    out = model(g, v.float(), src_key_padding_mask=m)["cell_emb"]
                outs.append(out.cpu())
                del G, V, M, g, v, m, out
                torch.cuda.empty_cache()
                gc.collect()
                i += current_bs
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                gc.collect()
                if current_bs <= min_bs:
                    raise RuntimeError(f"OOM even at batch size {current_bs}.")
                current_bs = max(min_bs, current_bs // 2)
    return torch.cat(outs, dim=0)


In [8]:
# ================================================================================
# STEP 1: DATA ENGINEERING/ANALYSIS
# ================================================================================
print(f"{'='*80}")
print(f"STEP 1: DATA ENGINEERING/ANALYSIS")
print(f"{'='*80}")
print(f"\nCell type: {config['celltype']}")
print(f"LAM cells: {len(idx_lam)}")
print(f"Control cells: {len(idx_ctrl)}")
pad_id = vocab["<pad>"]
print(f"\n{'='*80}")
print(f"BASELINE EMBEDDING COMPUTATION")
print(f"{'='*80}")
print(f"Baseline is computed over all genes in the candidate gene set")
print(f"\n[EMBEDDING] LAM cells ({len(idx_lam)} cells)...")
# Print gene sequence shape info for first batch
toks_sample = [TOKENS[j] for j in idx_lam[:5]]
for i, t in enumerate(toks_sample):
    print(f"  Cell {i}: {len(t['genes'])} genes (seq length {len(t['genes'])})")
emb_lam = embed(idx_lam)
print(f"  ✓ emb_lam shape: {emb_lam.shape}")
print(f"  ✓ dtype: {emb_lam.dtype}")
print(f"  ✓ device: {emb_lam.device}")
print(f"  ✓ memory: {emb_lam.element_size() * emb_lam.nelement() / 1e9:.2f} GB")

print(f"\n[EMBEDDING] Control cells ({len(idx_ctrl)} cells)...")
toks_sample = [TOKENS[j] for j in idx_ctrl[:5]]
for i, t in enumerate(toks_sample):
    print(f"  Cell {i}: {len(t['genes'])} genes (seq length {len(t['genes'])})")
emb_ctrl = embed(idx_ctrl)
print(f"  ✓ emb_ctrl shape: {emb_ctrl.shape}")
print(f"  ✓ dtype: {emb_ctrl.dtype}")
print(f"  ✓ device: {emb_ctrl.device}")
print(f"  ✓ memory: {emb_ctrl.element_size() * emb_ctrl.nelement() / 1e9:.2f} GB")


STEP 1: DATA ENGINEERING/ANALYSIS

Cell type: LEC
LAM cells: 697
Control cells: 7

BASELINE EMBEDDING COMPUTATION
Baseline is computed over all genes in the candidate gene set

[EMBEDDING] LAM cells (697 cells)...
  Cell 0: 1201 genes (seq length 1201)
  Cell 1: 1201 genes (seq length 1201)
  Cell 2: 1201 genes (seq length 1201)
  Cell 3: 1201 genes (seq length 1201)
  Cell 4: 1201 genes (seq length 1201)
  ✓ emb_lam shape: torch.Size([697, 512])
  ✓ dtype: torch.float32
  ✓ device: cpu
  ✓ memory: 0.00 GB

[EMBEDDING] Control cells (7 cells)...
  Cell 0: 1201 genes (seq length 1201)
  Cell 1: 1201 genes (seq length 1201)
  Cell 2: 1201 genes (seq length 1201)
  Cell 3: 1201 genes (seq length 1201)
  Cell 4: 1201 genes (seq length 1201)
  ✓ emb_ctrl shape: torch.Size([7, 512])
  ✓ dtype: torch.float32
  ✓ device: cpu
  ✓ memory: 0.00 GB


In [9]:

print(f"\n===== COMPUTING CENTROIDS =====")
print(f"LAM centroid shape: {emb_lam.mean(0).shape}")
print(f"Control centroid shape: {emb_ctrl.mean(0).shape}")

print(f"\n===== DEFINE FIXED HEALTHY TARGET =====")
fixed_centroid_control = emb_ctrl.mean(0)
print(f"Fixed control centroid shape: {fixed_centroid_control.shape}")

print(f"\n===== PER-CELL METRICS =====")
print(f"Computing cosine similarity for {len(emb_lam)} LAM cells...")
per_cell_cosine_to_control = torch.nn.functional.cosine_similarity(
    emb_lam, fixed_centroid_control.unsqueeze(0), dim=1
)
print(f"Per-cell cosine similarity shape: {per_cell_cosine_to_control.shape}")
print(f"Per-cell cosine stats: min={per_cell_cosine_to_control.min():.4f}, max={per_cell_cosine_to_control.max():.4f}, mean={per_cell_cosine_to_control.mean():.4f}")

print(f"\nComputing L2 distance for {len(emb_lam)} LAM cells...")
per_cell_l2_to_control = torch.norm(
    emb_lam - fixed_centroid_control, dim=1
)
print(f"Per-cell L2 distance shape: {per_cell_l2_to_control.shape}")
print(f"Per-cell L2 stats: min={per_cell_l2_to_control.min():.4f}, max={per_cell_l2_to_control.max():.4f}, mean={per_cell_l2_to_control.mean():.4f}")

print(f"\n===== POPULATION-LEVEL METRICS =====")
print(f"Computing aggregate cosine and L2...")
mean_cosine_lam_to_control = per_cell_cosine_to_control.mean()
mean_l2_lam_to_control = per_cell_l2_to_control.mean()
print(f"Mean cosine (LAM → Control): {mean_cosine_lam_to_control:.4f}")
print(f"Mean L2 (LAM → Control): {mean_l2_lam_to_control:.4f}")

print(f"\n===== CENTROID-LEVEL METRICS =====")
print(f"Computing centroid comparisons...")
centroid_lam = emb_lam.mean(0)
print(f"LAM centroid shape: {centroid_lam.shape}")
centroid_cosine_lam_vs_control = torch.nn.functional.cosine_similarity(
    centroid_lam.unsqueeze(0), fixed_centroid_control.unsqueeze(0)
)
print(f"Centroid cosine (LAM → Control): {centroid_cosine_lam_vs_control.item():.4f}")
centroid_l2_lam_vs_control = torch.norm(
    centroid_lam - fixed_centroid_control
)
print(f"Centroid L2 (LAM → Control): {centroid_l2_lam_vs_control.item():.4f}")

timestamp_step1 = time.strftime("%Y-%m-%d %H:%M:%S")
print(f"\n{'='*80}")
print(f"STEP 1 is completed")
print(f"Timestamp: {timestamp_step1}")
print(f"{'='*80}")



===== COMPUTING CENTROIDS =====
LAM centroid shape: torch.Size([512])
Control centroid shape: torch.Size([512])

===== DEFINE FIXED HEALTHY TARGET =====
Fixed control centroid shape: torch.Size([512])

===== PER-CELL METRICS =====
Computing cosine similarity for 697 LAM cells...
Per-cell cosine similarity shape: torch.Size([697])
Per-cell cosine stats: min=0.9976, max=1.0000, mean=0.9997

Computing L2 distance for 697 LAM cells...
Per-cell L2 distance shape: torch.Size([697])
Per-cell L2 stats: min=0.0385, max=1.5564, mean=0.4517

===== POPULATION-LEVEL METRICS =====
Computing aggregate cosine and L2...
Mean cosine (LAM → Control): 0.9997
Mean L2 (LAM → Control): 0.4517

===== CENTROID-LEVEL METRICS =====
Computing centroid comparisons...
LAM centroid shape: torch.Size([512])
Centroid cosine (LAM → Control): 1.0000
Centroid L2 (LAM → Control): 0.0171

STEP 1 is completed
Timestamp: 2026-05-26 12:16:05


In [10]:
# ================================================================================
# STEP 2: LOAD CANDIDATE GENES AND ANALYZE AVAILABILITY
# ================================================================================
print(f"\n{'='*80}")
print(f"STEP 2: LOAD CANDIDATE GENES AND ANALYZE AVAILABILITY")
print(f"{'='*80}")

print(f"\n[MAPPING] Using existing token-to-gene mappings...")
print(f"Token-to-gene mapping: {len(token_to_gene)} tokens mapped")
print(f"Gene-to-token mapping: {len(gene_to_token)} genes mapped")

print(f"\n[LOADING] Reading candidate genes from file...")
candidate_genes_file = Path("../data/candidate_genes/test.txt")
with open(candidate_genes_file, 'r') as f:
    candidate_gene_symbols = [line.strip() for line in f if line.strip()]
print(f"Loaded {len(candidate_gene_symbols)} candidate genes from file")

print(f"\n[FUNCTION] check_gene_availability()")
def check_gene_availability(candidate_genes, cell_indices):
    """Count how many cells express each candidate gene and calculate avg expression"""
    gene_availability = {}
    gene_avg_expr = {}
    for gene_symbol in candidate_genes:
        gene_token_id = gene_to_token.get(gene_symbol, None)
        if gene_token_id is None:
            gene_availability[gene_symbol] = 0
            gene_avg_expr[gene_symbol] = 0.0
            continue
        
        count = 0
        total_expr = 0.0
        for cell_idx in cell_indices:
            tok = TOKENS[cell_idx]
            genes = tok["genes"]
            values = tok["values"]
            gene_mask = (genes == gene_token_id)
            if gene_mask.any() and (values[gene_mask] > 0).any():
                count += 1
                expr_val = values[gene_mask][0].item() if hasattr(values[gene_mask][0], 'item') else values[gene_mask][0]
                total_expr += float(expr_val)
        gene_availability[gene_symbol] = count
        gene_avg_expr[gene_symbol] = total_expr / count if count > 0 else 0.0
    return gene_availability, gene_avg_expr

print(f"\n[CHECKING] Gene availability in LAM cells...")
lam_gene_availability, lam_gene_avg_expr = check_gene_availability(candidate_gene_symbols, idx_lam)
print(f"[CHECKING] Gene availability in Control cells...")
ctrl_gene_availability, ctrl_gene_avg_expr = check_gene_availability(candidate_gene_symbols, idx_ctrl)

print(f"\n{'='*80}")
print(f"CANDIDATE GENE AVAILABILITY TABLE")
print(f"{'='*80}")
print(f"{'Gene':<30} {'#LAM':<12} {'Avg_LAM':<12} {'#Control':<12} {'Avg_Ctrl':<12} {'#Total':<12}")
print(f"-" * 90)

for gene_symbol in candidate_gene_symbols:
    lam_count = lam_gene_availability.get(gene_symbol, 0)
    lam_avg = lam_gene_avg_expr.get(gene_symbol, 0.0)
    ctrl_count = ctrl_gene_availability.get(gene_symbol, 0)
    ctrl_avg = ctrl_gene_avg_expr.get(gene_symbol, 0.0)
    total_count = lam_count + ctrl_count
    print(f"{gene_symbol:<30} {lam_count:<12} {lam_avg:<12.4f} {ctrl_count:<12} {ctrl_avg:<12.4f} {total_count:<12}")

print(f"\n{'='*80}")
available_genes = sum(1 for g in candidate_gene_symbols if lam_gene_availability.get(g, 0) > 0 or ctrl_gene_availability.get(g, 0) > 0)
print(f"Summary: {available_genes}/{len(candidate_gene_symbols)} genes available in data")
print(f"{'='*80}")

print(f"\n[FUNCTION] get_gene_expression_per_cell()")
def get_gene_expression_per_cell(adata_subset_indices):
    gene_counts_per_cell = {}
    for cell_idx in adata_subset_indices:
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        nonzero = (values > 0).sum().item() if hasattr(values, 'sum') else sum(v > 0 for v in values)
        gene_counts_per_cell[cell_idx] = nonzero
    return gene_counts_per_cell

print(f"\n[FUNCTION] count_gene_cells()")
def count_gene_cells(adata_subset_indices, gene_threshold=10):
    gene_cell_count = {}
    for cell_idx in adata_subset_indices:
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        for g, v in zip(genes, values):
            if v > 0:
                gene_name = token_to_gene.get(int(g), None)
                if gene_name is None:
                    continue
                if gene_name not in gene_cell_count:
                    gene_cell_count[gene_name] = 0
                gene_cell_count[gene_name] += 1
    filtered_genes = {g: c for g, c in gene_cell_count.items() if c > gene_threshold}
    return filtered_genes


print(f"\n===== FILTERING CANDIDATE GENES =====")
print(f"Checking which genes exist in token vocabulary...")

# Filter genes that exist in token vocabulary
available_in_vocab = []
missing_in_vocab = []

for gene_symbol in candidate_gene_symbols:
    if gene_to_token.get(gene_symbol, None) is not None:
        available_in_vocab.append(gene_symbol)
    else:
        missing_in_vocab.append(gene_symbol)

print(f"\n  ✓ Genes in token vocabulary: {len(available_in_vocab)}")
print(f"  ✗ Genes NOT in vocabulary:  {len(missing_in_vocab)}")

if len(missing_in_vocab) > 0:
    print(f"\n  Missing genes ({len(missing_in_vocab)}):")
    for gene in missing_in_vocab[:10]:  # Show first 10
        print(f"    - {gene}")
    if len(missing_in_vocab) > 10:
        print(f"    ... and {len(missing_in_vocab) - 10} more")

# Use only genes that exist in vocabulary
candidate_gene_symbols = available_in_vocab

print(f"\n{'='*80}")
print(f"STEP 2: GENE ANALYSIS COMPLETE")
print(f"Genes for perturbation analysis: {len(candidate_gene_symbols)}")
print(f"Genes available in LAM cells: {sum(1 for g in candidate_gene_symbols if lam_gene_availability.get(g, 0) > 0)}")
print(f"Genes available in Control cells: {sum(1 for g in candidate_gene_symbols if ctrl_gene_availability.get(g, 0) > 0)}")
print(f"{'='*80}")

print(f"\n{'='*80}")
print(f"BATCH PROCESSING INFORMATION")
print(f"{'='*80}")
total_candidate_genes = len(candidate_gene_symbols)
num_batches = (total_candidate_genes + 99) // 100
print(f"Total candidate genes: {total_candidate_genes}")
print(f"Batch size: 100 genes per batch")
print(f"Number of batches: {num_batches}")
print(f"{'='*80}")

# Replace lam_sorted_genes with candidate genes as tuples for compatibility
lam_sorted_genes = [(gene, 0.0, 0.0, 0.0, 0.0, 0.0) for gene in candidate_gene_symbols]



STEP 2: LOAD CANDIDATE GENES AND ANALYZE AVAILABILITY

[MAPPING] Using existing token-to-gene mappings...
Token-to-gene mapping: 60697 tokens mapped
Gene-to-token mapping: 60697 genes mapped

[LOADING] Reading candidate genes from file...
Loaded 9 candidate genes from file

[FUNCTION] check_gene_availability()

[CHECKING] Gene availability in LAM cells...
[CHECKING] Gene availability in Control cells...

CANDIDATE GENE AVAILABILITY TABLE
Gene                           #LAM         Avg_LAM      #Control     Avg_Ctrl     #Total      
------------------------------------------------------------------------------------------
gene                           0            0.0000       0            0.0000       0           
LYVE1                          349          26.2378      2            31.5000      351         
PMEL                           4            14.5000      1            7.0000       5           
MDK                            311          23.1897      2            34.0000     

In [11]:
print(adata.obs)

                                           orig.ident  nCount_RNA  \
__________AAACCTGAGCTATGCT              SeuratProject      5477.0   
__________ACTGTCCAGACCTTTG              SeuratProject      4204.0   
__________TGACTTTGTCTAGCGC              SeuratProject      6191.0   
_______AML1166_Normal_AAAGATGAGAGTACAT  SeuratProject      4257.0   
_______AML1166_Normal_AACTCCCCATTCTTAC  SeuratProject       837.0   
...                                               ...         ...   
AML1172_Tumor_TTCTTAGAGGATCGCA          SeuratProject      4804.0   
AML1172_Tumor_TTCTTAGTCTCTAAGG          SeuratProject      3999.0   
AML1172_Tumor_TTGGCAAGTGATGATA          SeuratProject      2218.0   
AML1172_Tumor_TTGTAGGAGCACAGGT          SeuratProject       766.0   
AML1172_Tumor_TTTGGTTAGCCCTAAT          SeuratProject      1132.0   

                                        nFeature_RNA              ID disease  \
__________AAACCTGAGCTATGCT                      2135         AML1098     AML   
__________A

In [12]:
# ================================================================================
# STEP 3: GENE PERTURBATION AND SHIFT COMPUTATION
# ================================================================================
print(f"\n{'='*80}")
print(f"STEP 3: GENE PERTURBATION AND SHIFT COMPUTATION")
print(f"{'='*80}")

timestamp_step3_start = time.strftime("%Y-%m-%d %H:%M:%S")
print(f"\nSTEP 3 Start: {timestamp_step3_start}")

results_dir = Path(save_dir)
results_dir.mkdir(parents=True, exist_ok=True)
csv_dir = results_dir / "csvs"
csv_dir.mkdir(exist_ok=True)
print(f"\n[FOLDER] Created results directory: {results_dir}")

perturbed_output_file = csv_dir / f"KO_TF_{config['celltype']}_perturbed_genes_analysis.csv"
print(f"[FILE] Creating dynamic output: {perturbed_output_file}")

print(f"\n[FUNCTION] generate_final_figure()")
def generate_final_figure(csv_path, results_dir):
    import pandas as pd
    csv_data = pd.read_csv(csv_path)
    if len(csv_data) == 0:
        return
    gene_names = csv_data["gene"].values
    l2_shifts = csv_data["mean_l2_shift"].values
    cosine_shifts = csv_data["mean_cosine_shift"].values
    centroid_shifts = csv_data["centroid_shift"].values
    top_100_idx = np.argsort(l2_shifts)[::-1][:100]
    gene_names_top = gene_names[top_100_idx]
    l2_shifts_top = l2_shifts[top_100_idx]
    cosine_shifts_top = cosine_shifts[top_100_idx]
    centroid_shifts_top = centroid_shifts[top_100_idx]
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))
    fig.suptitle(f"{config['celltype']} - Top 100 Gene Perturbation Effects", fontsize=16, fontweight='bold')
    sorted_idx_l2 = np.argsort(l2_shifts_top)[::-1]
    axes[0].barh(range(len(sorted_idx_l2)), l2_shifts_top[sorted_idx_l2], color='steelblue')
    axes[0].set_yticks(range(len(sorted_idx_l2)))
    axes[0].set_yticklabels(gene_names_top[sorted_idx_l2], fontsize=7)
    axes[0].set_xlabel('Mean L2 Shift', fontsize=11)
    axes[0].set_title('Top L2 Distance Shift', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].grid(axis='x', alpha=0.3)
    sorted_idx_cos = np.argsort(cosine_shifts_top)
    axes[1].barh(range(len(sorted_idx_cos)), cosine_shifts_top[sorted_idx_cos], color='coral')
    axes[1].set_yticks(range(len(sorted_idx_cos)))
    axes[1].set_yticklabels(gene_names_top[sorted_idx_cos], fontsize=7)
    axes[1].set_xlabel('Mean Cosine Shift', fontsize=11)
    axes[1].set_title('Top Cosine Similarity Shift', fontsize=12, fontweight='bold')
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', alpha=0.3)
    sorted_idx_centroid = np.argsort(centroid_shifts_top)[::-1]
    axes[2].barh(range(len(sorted_idx_centroid)), centroid_shifts_top[sorted_idx_centroid], color='mediumseagreen')
    axes[2].set_yticks(range(len(sorted_idx_centroid)))
    axes[2].set_yticklabels(gene_names_top[sorted_idx_centroid], fontsize=7)
    axes[2].set_xlabel('Centroid Shift', fontsize=11)
    axes[2].set_title('Top Centroid Shift', fontsize=12, fontweight='bold')
    axes[2].invert_yaxis()
    axes[2].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    (results_dir / "figure").mkdir(exist_ok=True)
    fig_path = (results_dir / "figure") / f"{config['celltype']}_gene_ranking_analysis.png"
    plt.savefig(str(fig_path), dpi=150, bbox_inches='tight')
    plt.close()

print(f"\n[FUNCTION] build_perturbed_matrix()")
def build_perturbed_matrix(gene_name, cell_indices, median_expr_threshold=None):
    gene_token_id = gene_to_token.get(gene_name, None)
    if gene_token_id is None:
        print(f"WARNING: Gene {gene_name} not found in token mapping!")
        return None, None
    
    if median_expr_threshold is None:
        median_expr_threshold = 0
    
    perturbed_genes_list = []
    high_expr_cell_indices = []
    
    for batch_idx, cell_idx in enumerate(cell_indices):
        if (batch_idx + 1) % 200 == 0 or batch_idx == len(cell_indices) - 1:
            print(f"{batch_idx + 1}/{len(cell_indices)}")
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        gene_mask = (genes == gene_token_id)
        
        if gene_mask.any():
            gene_value = values[gene_mask][0].item() if hasattr(values[gene_mask][0], 'item') else values[gene_mask][0]
            
            # MEDIAN FILTER REMOVED - perturb all cells expressing the gene
            # if gene_value > median_expr_threshold:
            if True:  # Perturb all expressing cells
                # TRUE KNOCKOUT: Remove gene token completely from 32k space
                keep_mask = genes != gene_token_id
                new_genes = genes[keep_mask]
                new_values = values[keep_mask]
                
                # Sanity check: gene count should should decrease by 1 (print every 200 cells)
                if (batch_idx + 1) % 200 == 0 or batch_idx == len(cell_indices) - 1:
                    print(f"Gene tokens: {len(genes)} -> {len(new_genes)} (removed 1)")
                
                perturbed_genes_list.append((new_genes, new_values))
                high_expr_cell_indices.append(cell_idx)
    
    return perturbed_genes_list, high_expr_cell_indices

print(f"\n[FUNCTION] embed_perturbed_tokens()")
def embed_perturbed_tokens(perturbed_genes_list, bs=64):
    outs = []
    i = 0
    while i < len(perturbed_genes_list):
        batch_idx_range = min(bs, len(perturbed_genes_list) - i)
        batch_toks = perturbed_genes_list[i:i+batch_idx_range]
        # Use all genes in the perturbed cell (Whole Genome minus knockout)
        max_len = max(len(t[0]) for t in batch_toks)
        G = torch.full((len(batch_toks), max_len), pad_id, dtype=torch.long)
        V = torch.zeros((len(batch_toks), max_len), dtype=torch.float16)
        for k, (genes, values) in enumerate(batch_toks):
            L = len(genes)
            if L > 0:
                G[k, :L] = genes
                V[k, :L] = values if values.dtype == torch.float16 else values.to(torch.float16)
        M = (G == pad_id)
        g = G.to(device, non_blocking=True)
        v = V.to(device, non_blocking=True)
        m = M.to(device, non_blocking=True)
        with torch.no_grad(), torch.autocast(device_type="cuda"):
            out = model(g, v.float(), src_key_padding_mask=m)["cell_emb"]
        outs.append(out.cpu())
        del G, V, M, g, v, m, out
        torch.cuda.empty_cache()
        i += batch_idx_range
    return torch.cat(outs, dim=0)

print(f"\n===== HELPER FUNCTION: Bootstrap p-value =====")
def bootstrap_pvalue(shifts, n_perms=1000):
    # Convert to rescue direction: negative L2 shift is good (cells moving toward control)
    observed = -np.mean(shifts)
    null_means = []
    for _ in range(n_perms):
        idx = np.random.choice(len(shifts), size=len(shifts), replace=True)
        null_means.append(-np.mean(shifts[idx]))
    p_val = (np.sum(np.array(null_means) >= observed) + 1) / (n_perms + 1)
    return p_val

print(f"\n===== HELPER FUNCTION: Find cells expressing gene =====")
def find_cells_expressing_gene(gene_name, cell_indices_subset):
    gene_token_id = gene_to_token.get(gene_name, None)
    if gene_token_id is None:
        return []
    expressing_cells = []
    for cell_idx in cell_indices_subset:
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        mask = (genes == gene_token_id)
        if mask.any() and (values[mask] > 0).any():
            expressing_cells.append(cell_idx)
    return expressing_cells

ctrl_centroid = emb_ctrl.mean(0)
baseline_lam_mean = emb_lam.mean(0)

print(f"\n===== DONOR MAPPING (DonorID) =====")
DONOR_COL = "ID"
donor_ids = adata.obs[DONOR_COL].iloc[idx_lam].values
idx_lam_donors = {int(idx): donor for idx, donor in zip(idx_lam, donor_ids)}
print(f"Unique donors: {len(set(donor_ids))}")
print(f"Donors: {sorted(set(donor_ids))}")

total_genes = len(lam_sorted_genes)
gene_count = 0
batch_num = 1
batch_start = 0

for gene_idx, (gene_name, expr_val, delta_pct, pct_lam, pct_ctrl, rank_score) in enumerate(lam_sorted_genes):
    batch_position = gene_idx - batch_start + 1
    genes_in_batch = min(100, total_genes - batch_start)
    
    if batch_position == 1:
        batch_end = min(batch_start + 100, total_genes)
        print(f"\n{'='*80}")
        print(f"===== PERTURBING {batch_start}-{batch_end-1} LAM GENES / BATCH {batch_num} =====")
        print(f"Total genes in this batch: {genes_in_batch}")
        print(f"Total genes in dataset: {total_genes}")
        print(f"{'='*80}")
    
    cell_indices = find_cells_expressing_gene(gene_name, idx_lam)
    if len(cell_indices) == 0:
        continue
    
    # ===== STATIC EXPRESSION CONTEXT =====
    gene_token_id = gene_to_token.get(gene_name, None)
    lam_expr_values = []
    ctrl_expr_values = []
    
    for cell_idx in idx_lam:
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        mask = (genes == gene_token_id)
        expr_val = float(values[mask].mean().item()) if mask.any() else 0.0
        lam_expr_values.append(expr_val)
    
    for cell_idx in idx_ctrl:
        tok = TOKENS[cell_idx]
        genes = tok["genes"]
        values = tok["values"]
        mask = (genes == gene_token_id)
        expr_val = float(values[mask].mean().item()) if mask.any() else 0.0
        ctrl_expr_values.append(expr_val)
    
    mean_expr_lam = float(np.mean(lam_expr_values))
    mean_expr_ctrl = float(np.mean(ctrl_expr_values))
    delta_expr = mean_expr_lam - mean_expr_ctrl
    
    eps = 1e-6
    log2FC = float(np.log2((mean_expr_lam + eps) / (mean_expr_ctrl + eps)))
    
    # Compute median expression threshold for this gene in LAM cells
    # ONLY among cells that express the gene (exclude zeros)
    expressing_values = [v for v in lam_expr_values if v > 0]
    if len(expressing_values) > 0:
        median_expr_threshold = float(np.median(expressing_values))
        mean_expr_lam_active = float(np.mean(expressing_values))
    else:
        median_expr_threshold = 0.0
        mean_expr_lam_active = 0.0
    
    # Compute prevalence: fraction of LAM cells expressing this gene
    pct_expr_lam = float(len(expressing_values) / len(lam_expr_values)) if len(lam_expr_values) > 0 else 0.0
    
    timestamp_gene = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n[GENE {batch_position}/{genes_in_batch}] ({gene_count + 1}/{total_genes}) {gene_name} | {timestamp_gene}")
    lam_count = lam_gene_availability.get(gene_name, 0)
    ctrl_count = ctrl_gene_availability.get(gene_name, 0)
    print(f"1/{len(candidate_gene_symbols)} | This gene is in {lam_count} LAM cells, {ctrl_count} Control cells")
    print(f"2/{len(candidate_gene_symbols)} | Starting perturbation of {lam_count} LAM cells...")
    
    perturbed_genes_list, cell_idx_list = build_perturbed_matrix(gene_name, cell_indices, median_expr_threshold=median_expr_threshold)
    if perturbed_genes_list is None:
        continue
    
    print(f"\nNEW embedding after perturbation for {gene_name} in {len(cell_idx_list)} cells is:")
    # Print perturbed gene sequence shape info for first batch
    if len(perturbed_genes_list) > 0:
        print(f"  First perturbed cell: {len(perturbed_genes_list[0][0])} genes (seq length {len(perturbed_genes_list[0][0])})")
    perturbed_emb = embed_perturbed_tokens(perturbed_genes_list, bs=64)
    print(f"  ✓ perturbed_emb shape: {perturbed_emb.shape}")
    print(f"{perturbed_emb.shape}")
    
    # Use cell_idx_list (post-filter cells from build_perturbed_matrix) instead of cell_indices (pre-filter)
    local_pos = torch.tensor([idx_lam_to_pos[int(idx)] for idx in cell_idx_list])
    baseline_emb = emb_lam[local_pos]
    perturbed_emb_gpu = perturbed_emb.to(device)
    baseline_emb_gpu = baseline_emb.to(device)
    ctrl_centroid_gpu = ctrl_centroid.to(device)
    baseline_lam_mean_gpu = baseline_lam_mean.to(device)
    
    l2_baseline = torch.norm(baseline_emb_gpu - ctrl_centroid_gpu, dim=1)
    l2_perturbed = torch.norm(perturbed_emb_gpu - ctrl_centroid_gpu, dim=1)
    l2_shifts = (l2_perturbed - l2_baseline).cpu().numpy()
    
    cos_baseline = torch.nn.functional.cosine_similarity(baseline_emb_gpu, ctrl_centroid_gpu.unsqueeze(0), dim=1)
    cos_perturbed = torch.nn.functional.cosine_similarity(perturbed_emb_gpu, ctrl_centroid_gpu.unsqueeze(0), dim=1)
    cosine_shifts = (cos_perturbed - cos_baseline).cpu().numpy()
    
    # Compute embedding disruption: mean L2 distance between perturbed and baseline embeddings
    delta_emb = torch.norm(perturbed_emb_gpu - baseline_emb_gpu, dim=1)
    mean_delta_emb = float(delta_emb.mean().item())
    
    # CORRECT CENTROID REPLACEMENT: Clone full LAM embedding, replace only perturbed cells
    emb_lam_modified = emb_lam.clone().to(device)  # Full LAM embedding
    emb_lam_modified[local_pos] = perturbed_emb_gpu  # Replace perturbed cells
    
    # Sanity check: shapes must match
    assert emb_lam_modified.shape == emb_lam.shape, f"Shape mismatch: {emb_lam_modified.shape} vs {emb_lam.shape}"
    
    perturbed_centroid = emb_lam_modified.mean(0)  # Compute centroid from mixed embedding
    centroid_shift = torch.linalg.norm(perturbed_centroid - baseline_lam_mean_gpu).item()
    del emb_lam_modified  # Clean up GPU memory
    
    l2_baseline_np = l2_baseline.cpu().numpy()
    l2_perturbed_np = l2_perturbed.cpu().numpy()
    cos_baseline_np = cos_baseline.cpu().numpy()
    cos_perturbed_np = cos_perturbed.cpu().numpy()
    
    mean_l2_shift = float(l2_shifts.mean())
    mean_cosine_shift = float(cosine_shifts.mean())
    
    # RESCUE SCORE: Population-level centroid shift (entire LAM population)
    # Not just the perturbed cells, but how the full population centroid moves
    baseline_centroid_to_control_l2 = float(torch.norm(baseline_lam_mean_gpu - ctrl_centroid_gpu).item())
    perturbed_centroid_to_control_l2 = float(torch.norm(perturbed_centroid - ctrl_centroid_gpu).item())
    rescue_score = baseline_centroid_to_control_l2 - perturbed_centroid_to_control_l2
    
    # NORMALIZED RESCUE: Efficiency metric (rescue score per unit of embedding disruption)
    # High normalized_rescue = therapeutic improvement without breaking the system
    normalized_rescue = float(rescue_score / (mean_delta_emb + 1e-6))
    
    p_value_l2 = bootstrap_pvalue(l2_shifts, n_perms=1000)
    
    effect_size_l2 = float((l2_perturbed_np.mean() - l2_baseline_np.mean()) / (l2_baseline_np.std() + 1e-8))
    
    frac_improved_l2 = float(np.mean(l2_perturbed_np < l2_baseline_np))
    n_cells_improved = int((l2_perturbed_np < l2_baseline_np).sum())
    frac_improved_cos = float(np.mean(cos_perturbed_np > cos_baseline_np))
    
    is_weak_effect = frac_improved_l2 < 0.3
    
    donor_shifts = []
    for donor in set(idx_lam_donors.values()):
        donor_cell_indices = [c for c in cell_idx_list if int(c) in idx_lam_donors and idx_lam_donors[int(c)] == donor]
        if len(donor_cell_indices) >= 2:
            # Map donor_cell_indices to positions in cell_idx_list (post-filter)
            donor_indices_in_filtered = [i for i, idx in enumerate(cell_idx_list) if idx in donor_cell_indices]
            if len(donor_indices_in_filtered) > 0:
                donor_l2_shift = l2_shifts[donor_indices_in_filtered]
                if len(donor_l2_shift) > 0:
                    donor_shifts.append(donor_l2_shift.mean())
    
    donor_mean_l2 = float(np.mean(donor_shifts)) if len(donor_shifts) > 0 else 0.0
    donor_std_l2 = float(np.std(donor_shifts)) if len(donor_shifts) > 1 else 0.0
    
    n_cells_value_gt_0 = len(cell_idx_list)
    pct_lam_perturbed = float(100.0 * len(cell_idx_list) / len(idx_lam))
    median_l2_shift = float(np.median(l2_shifts))
    std_l2_shift = float(np.std(l2_shifts))
    
    # delta_centroid_to_control = how the centroid moved (perturbed - baseline)
    # Note: rescue_score = baseline - perturbed, so rescue_score = -delta_centroid_to_control
    delta_centroid_to_control = float(perturbed_centroid_to_control_l2 - baseline_centroid_to_control_l2)
    
    ctrl_cell_indices = find_cells_expressing_gene(gene_name, idx_ctrl)
    pct_ctrl_expressing = float(100.0 * len(ctrl_cell_indices) / len(idx_ctrl))
    
    result = {
        "gene": gene_name,
        "normalized_rescue": normalized_rescue,
        "rescue_score": rescue_score,
        "mean_delta_emb": mean_delta_emb,
        "frac_improved_l2": frac_improved_l2,
        "n_cells_improved": n_cells_improved,
        "mean_l2_shift": mean_l2_shift,
        "delta_centroid_to_control": delta_centroid_to_control,
        "p_value_l2": float(p_value_l2),
        "mean_expr_lam_active": mean_expr_lam_active,
        "pct_expr_lam": pct_expr_lam,
        "log2FC": log2FC,
        "n_cells": len(cell_idx_list),
        "pct_lam_perturbed": pct_lam_perturbed,
        "median_l2_shift": median_l2_shift,
        "std_l2_shift": std_l2_shift,
        "donor_mean_l2_shift": donor_mean_l2,
        "donor_std_l2_shift": donor_std_l2,
        "mean_cosine_shift": mean_cosine_shift,
        "frac_improved_cos": frac_improved_cos,
        "n_cells_value_gt_0": n_cells_value_gt_0,
        "baseline_centroid_to_control": baseline_centroid_to_control_l2,
        "perturbed_centroid_to_control": perturbed_centroid_to_control_l2,
        "is_weak_effect": int(is_weak_effect),
        "mean_expr_lam": mean_expr_lam,
        "mean_expr_ctrl": mean_expr_ctrl,
        "delta_expr": delta_expr,
        "pct_ctrl_expressing": pct_ctrl_expressing,
        "centroid_shift": float(centroid_shift),
        "effect_size_l2": effect_size_l2,
    }
    
    print(f"\n{'─'*80}")
    print(f"RESULTS FOR: {gene_name}")
    print(f"{'─'*80}")
    
    # Determine quality indicator
    quality = "🔥 STRONG" if rescue_score > 0.1 and frac_improved_l2 > 0.6 else "✓ MODERATE" if rescue_score > 0 else "✗ WEAK"
    significance = "***" if p_value_l2 < 0.01 else "**" if p_value_l2 < 0.05 else "*" if p_value_l2 < 0.1 else "ns"
    
    print(f"\n  📊 THERAPEUTIC POTENTIAL: {quality}")
    print(f"     rescue_score = {rescue_score:+.4f} {significance}")
    print(f"     normalized_rescue = {normalized_rescue:+.4f} (rescue per unit disruption)")
    print(f"     mean_delta_emb = {mean_delta_emb:.4f} (embedding disruption)")
    print(f"     (positive = moves population toward healthy control)")
    print(f"     is_weak_effect = {int(is_weak_effect)}")
    
    print(f"\n  🧬 EXPRESSION PROFILE:")
    print(f"     LAM avg expr (population) = {mean_expr_lam:.4f}")
    print(f"     LAM avg expr (active only) = {mean_expr_lam_active:.4f}")
    print(f"     LAM expr prevalence = {pct_expr_lam:.1%}")
    print(f"     Control avg expr  = {mean_expr_ctrl:.4f}")
    print(f"     Expression delta  = {delta_expr:+.4f}")
    print(f"     log2 fold-change  = {log2FC:+.4f}")
    
    print(f"\n  📍 CELL COVERAGE:")
    print(f"     Cells perturbed   = {n_cells_value_gt_0} / {len(idx_lam)} LAM ({pct_lam_perturbed:.1f}%)")
    print(f"     Control express   = {len(ctrl_cell_indices)} / {len(idx_ctrl)} ({pct_ctrl_expressing:.1f}%)")
    
    print(f"\n  📈 L2 DISTANCE SHIFTS:")
    print(f"     Mean shift        = {mean_l2_shift:+.4f}")
    print(f"     Median shift      = {median_l2_shift:+.4f}")
    print(f"     Std dev           = {std_l2_shift:.4f}")
    print(f"     Mean cosine shift = {mean_cosine_shift:+.4f}")
    print(f"     Cells improved    = {frac_improved_l2:.1%} ({n_cells_improved} cells)")
    print(f"     Cells analyzed    = {len(cell_idx_list)} / {len(idx_lam)} LAM")
    
    print(f"\n  🎯 POPULATION CENTROID SHIFT:")
    print(f"     Baseline → Control = {baseline_centroid_to_control_l2:.4f}")
    print(f"     Perturbed → Control = {perturbed_centroid_to_control_l2:.4f}")
    print(f"     Centroid moved by  = {abs(delta_centroid_to_control):.4f}")
    print(f"     Direction         = {'toward healthy ↓' if delta_centroid_to_control < 0 else 'away from healthy ↑'}")
    print(f"     Delta control     = {delta_centroid_to_control:+.4f}")
    
    print(f"\n  🔬 STATISTICAL CONFIDENCE:")
    print(f"     p-value (L2)      = {p_value_l2:.4f} {significance}")
    print(f"     Effect size       = {effect_size_l2:.4f}")
    print(f"     Cosine improved   = {frac_improved_cos:.1%}")
    print(f"     Donor consistency = μ={donor_mean_l2:.4f}, σ={donor_std_l2:.4f}")
    
    with open(str(perturbed_output_file), 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=result.keys())
        if gene_count == 0:
            writer.writeheader()
        writer.writerow(result)
    
    gene_count += 1
    batch_position_global = gene_idx + 1
    
    if batch_position_global % 100 == 0 or batch_position_global == total_genes:
        batch_end = min(batch_position_global, total_genes)
        batch_csv_file = csv_dir / f"{config['celltype']}_{batch_end}_genes.csv"
        import pandas as pd
        csv_data = pd.read_csv(perturbed_output_file)
        csv_data.to_csv(batch_csv_file, index=False)
        print(f"\n>>> Batch {batch_num} CSV saved: {config['celltype']}_{batch_end}_genes.csv")
        print(f"    Cumulative genes: {gene_count} | Total in file: {len(csv_data)}")
        
        if batch_position_global < total_genes:
            batch_num += 1
            batch_start += 100
    
    torch.cuda.empty_cache()

print(f"\n{'='*80}")
print(f"✓ STEP 3: GENE PERTURBATION COMPLETE")
print(f"{'='*80}")
print(f"  Genes analyzed:     {gene_count}/{total_genes}")
print(f"  Total batches:      {batch_num}")
timestamp_step3_end = time.strftime("%Y-%m-%d %H:%M:%S")
print(f"  Completion time:    {timestamp_step3_end}")
print(f"{'='*80}")

print(f"\n{'='*80}")
print(f"POST-PROCESSING: FDR CORRECTION & VISUALIZATION")
print(f"{'='*80}")
import pandas as pd
from scipy.stats import rankdata

print(f"\n[1/3] Loading results from CSV...")
csv_data = pd.read_csv(perturbed_output_file)
print(f"  ✓ Loaded {len(csv_data)} genes")

if 'p_value_l2' in csv_data.columns:
    print(f"\n[2/3] Computing FDR-corrected q-values (Benjamini-Hochberg)...")
    p_vals = csv_data['p_value_l2'].values
    n_tests = len(p_vals)
    sorted_idx = np.argsort(p_vals)
    sorted_p = p_vals[sorted_idx]
    q_vals = np.ones_like(p_vals)
    for i, idx in enumerate(sorted_idx):
        q_vals[idx] = min(1.0, sorted_p[i] * n_tests / (i + 1))
    csv_data['q_value_l2'] = q_vals
    csv_data.to_csv(perturbed_output_file, index=False)
    
    # Summary stats
    sig_threshold_05 = np.sum(q_vals < 0.05)
    sig_threshold_01 = np.sum(q_vals < 0.01)
    print(f"  ✓ Q-values computed")
    print(f"    - Significant at q<0.05: {sig_threshold_05} genes")
    print(f"    - Significant at q<0.01: {sig_threshold_01} genes")

print(f"\n[3/3] Generating final visualization...")
generate_final_figure(perturbed_output_file, results_dir)
fig_path = results_dir / "figure" / f"{config['celltype']}_gene_ranking_analysis.png"
print(f"  ✓ Figure saved: {fig_path.name}")
print(f"    - Top 100 genes ranked by L2 distance shift")
print(f"    - Top 100 genes ranked by cosine similarity shift")
print(f"    - Top 100 genes ranked by centroid shift")


print(f"\n{'='*80}")
print(f"✓✓✓ ANALYSIS COMPLETE ✓✓✓")
print(f"{'='*80}")
print(f"\n📁 OUTPUT FILES:")
print(f"   Main results:  {perturbed_output_file}")
print(f"   Visualization: {fig_path}")
#print(f"   Log file:      {log_file}")
print(f"\n📊 SUMMARY STATISTICS:")
print(f"   Total genes analyzed:   {len(csv_data)}")
print(f"   Genes with rescue>0:    {np.sum(csv_data['rescue_score'] > 0)}")
print(f"   Genes with rescue<0:    {np.sum(csv_data['rescue_score'] < 0)}")
print(f"   Significant (q<0.05):   {sig_threshold_05}")
print(f"   Highly sig (q<0.01):    {sig_threshold_01}")
print(f"\n🎯 TOP 5 THERAPEUTIC CANDIDATES (by rescue_score):")
top_5 = csv_data.nlargest(5, 'rescue_score')[['gene', 'rescue_score', 'frac_improved_l2', 'p_value_l2']]
for idx, (_, row) in enumerate(top_5.iterrows(), 1):
    print(f"   {idx}. {row['gene']:12s} | rescue={row['rescue_score']:+.4f} | improved={row['frac_improved_l2']:.1%} | p={row['p_value_l2']:.4f}")

print(f"\n{'='*80}")



STEP 3: GENE PERTURBATION AND SHIFT COMPUTATION

STEP 3 Start: 2026-05-26 12:16:06

[FOLDER] Created results directory: save/dev-May26-12-13
[FILE] Creating dynamic output: save/dev-May26-12-13/csvs/KO_TF_LEC_perturbed_genes_analysis.csv

[FUNCTION] generate_final_figure()

[FUNCTION] build_perturbed_matrix()

[FUNCTION] embed_perturbed_tokens()

===== HELPER FUNCTION: Bootstrap p-value =====

===== HELPER FUNCTION: Find cells expressing gene =====

===== DONOR MAPPING (DonorID) =====
Unique donors: 7
Donors: ['AML1098', 'AML1162', 'AML1166_AML', 'AML1167_AML1', 'AML1167_AML2', 'AML1171_Tumor', 'AML1172_Tumor']

===== PERTURBING 0-7 LAM GENES / BATCH 1 =====
Total genes in this batch: 8
Total genes in dataset: 8

[GENE 1/8] (1/8) LYVE1 | 2026-05-26 12:16:06
1/8 | This gene is in 349 LAM cells, 2 Control cells
2/8 | Starting perturbation of 349 LAM cells...
200/349
Gene tokens: 1201 -> 1200 (removed 1)
349/349
Gene tokens: 1201 -> 1200 (removed 1)

NEW embedding after perturbation for 

In [13]:
test_embed_adata = scg.tasks.embed_data(
    adata,
    config["load_model"],
    gene_col="gene_name",
    obs_to_save=adata.obs.columns.tolist(),  # optional arg, only for saving metainfo
    batch_size=config["batch_size"],
    return_new_adata=True,
)

scGPT - INFO - match 1200/1200 genes in vocabulary of size 60697.


/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 4249/4249 [01:00<00:00, 70.41it/s]
/users/wag9iz/.conda/envs/rtd_perturb/lib/python3.9/site-packages/anndata/_core/anndata.py:402: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [14]:
test_embed_adata

AnnData object with n_obs × n_vars = 135965 × 512
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'ID', 'disease', 'subject', 'percent.mito', 'integrated_snn_res.1.5', 'seurat_clusters', 'AnnoCellType', 'harmony_clusters', 'celltype', 'condition', 'str_batch', 'batch_id'